## Проверка data-pipeline


In [33]:
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.dataloaders import create_dataloaders, create_test_dataloader


## Preprocessing с recommended heuristics

Проверяем новый режим: к train добавляются только `cabinet` и `dressing_room`, а количество примеров добирается до среднего размера класса.


In [34]:
# Запускаем preprocessing с рекомендуемыми heuristics
# --skip-image-verify ускоряет проверку в ноутбуке: файлы проверяются по наличию, но не открываются через PIL
preprocess_cmd = [
    sys.executable,
    "-m",
    "src.preprocess_data",
    "--include-heuristics",
    "recommended",
    "--skip-image-verify",
]

result = subprocess.run(
    preprocess_cmd,
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(result.stdout)


Папка с обработанными данными: /Users/user/projects/Python/room_type_classifier/data/processed
Mapping классов: /Users/user/projects/Python/room_type_classifier/data/processed/class_mapping.json
Manifest preprocessing: /Users/user/projects/Python/room_type_classifier/data/processed/preprocessing_manifest.json
train: строк до=4562, строк после=4311
train: строк после heuristics=4632
train: включены heuristics=['cabinet', 'dressing_room']
train: целевой размер класса для heuristics=227
val: строк до=500, строк после=477
test: строк до=48003, строк после=48003
test: строк для предсказания=47791 из 48003



In [35]:
# Проверяем, что heuristics добрали нужные классы до среднего размера
train_df = pd.read_csv(processed_dir / "train_df.csv")
target_count = manifest["heuristics_target_count"]

class_source_counts = (
    train_df
    .groupby(["result", "source"])
    .size()
    .rename("count")
    .reset_index()
)

class_source_counts.sort_values(["result", "source"])


,result,source,count
0,0,train,248
1,1,train,199
2,2,train,247
3,3,train,249
4,4,train,251
5,5,heuristics_cabinet,153
6,5,train,74
7,6,train,215
8,7,train,255
9,8,train,255


In [36]:
# Создаём DataLoader на основе processed CSV и локальных изображений
train_loader, val_loader = create_dataloaders(
    batch_size=32,
    num_workers=0,
    image_size=224,
    use_weighted_sampling=False
)

test_loader = create_test_dataloader(
    batch_size=32,
    num_workers=0,
    image_size=224
)

print("train объектов:", len(train_loader.dataset))
print("val объектов:", len(val_loader.dataset))
print("test объектов для предсказания:", len(test_loader.dataset))


train объектов: 4632
val объектов: 477
test объектов для предсказания: 47791


In [37]:
# Берём первый batch из train и test
images, targets = next(iter(train_loader))
test_images, test_image_ids, test_item_ids = next(iter(test_loader))


In [38]:
# Проверяем формы batch
print("train images:", images.shape)
print("train targets:", targets.shape)
print("test images:", test_images.shape)
print("первые image_id из test:", list(test_image_ids[:5]))


train images: torch.Size([32, 3, 224, 224])
train targets: torch.Size([32])
test images: torch.Size([32, 3, 224, 224])
первые image_id из test: ['15822317371', '15948569108', '16030687230', '16033186338', '15430455676']


In [39]:
# Загружаем ResNet-18 с предобученными весами
model = resnet18(weights=ResNet18_Weights.DEFAULT)


In [40]:
# Меняем последний слой под наши 19 классов
num_classes = 19
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)


In [41]:
# Прогоняем batch через модель без обучения
model.eval()

with torch.no_grad():
    train_outputs = model(images)
    test_outputs = model(test_images)

print("train outputs:", train_outputs.shape)
print("test outputs:", test_outputs.shape)


train outputs: torch.Size([32, 19])
test outputs: torch.Size([32, 19])


In [42]:
# прверка балансировки
def check_loader_class_balance(loader, max_batches=None):
    class_counter = Counter()

    for batch_idx, (_, targets) in enumerate(loader):
        class_counter.update(targets.tolist())

        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    balance_df = (
        pd.DataFrame(
            class_counter.items(),
            columns=["class_id", "count"]
        )
        .sort_values("class_id")
        .reset_index(drop=True)
    )

    balance_df["ratio"] = balance_df["count"] / balance_df["count"].sum()

    return balance_df

In [43]:
train_balance_df = check_loader_class_balance(train_loader)
train_balance_df

,class_id,count,ratio
0,0,248,0.053541
1,1,199,0.042962
2,2,247,0.053325
3,3,249,0.053756
4,4,251,0.054188
5,5,227,0.049007
6,6,215,0.046416
7,7,255,0.055052
8,8,255,0.055052
9,9,254,0.054836
